# Making SIMSOPT GPU native: smoothed local-residual augmented Lagrangian

Select **Runtime > Change runtime type > GPU**, then run all cells. This qualification compares the same matrix-free local-residual augmented-Lagrangian program on JAX CPU and NVIDIA GPU. It uses a zero-preserving smooth residual transform, staged family-scale continuation, and an absolute-or-relative first-order safeguard. It records final CPU/GPU field and coil-constraint metrics and exports both designs for ParaView.

In [ ]:
import shutil
import subprocess

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is None:
    raise RuntimeError("No NVIDIA GPU is attached. Select Runtime > Change runtime type > T4 GPU, disconnect the old runtime, and reconnect.")
subprocess.run([nvidia_smi], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report
assert jax.config.jax_enable_x64, report
assert any(device.platform == "gpu" for device in jax.devices()), report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu/test_augmented_lagrangian.py", "tests/gpu/test_local_residual_augmented_lagrangian_benchmark.py", "tests/gpu/test_objective.py"], cwd=repo, check=True)

In [ ]:
artifact_root = Path("/content/simsopt-local-residual-al")
artifact_root.mkdir(exist_ok=True)
result_path = artifact_root / "local-residual-augmented-lagrangian.json"
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/benchmark_local_residual_augmented_lagrangian.py", "--problem", "engineering", "--max-outer-iterations", "8", "--max-inner-iterations", "300", "--mu-init", "10", "--mu-max", "1e12", "--tau", "2", "--gradient-tolerance", "1e-8", "--constraint-tolerance", "1e-6", "--target-relative-tolerance", "0.10", "--inner-stationarity-factor", "1", "--inner-stationarity-relative-tolerance", "0.01", "--maxcor", "100", "--maxls", "50", "--residual-scaling-policy", "family_l2", "--minimum-residual-scaling-policy", "sqrt_count", "--constraint-scale-reduction-factor", "0.5", "--constraint-transform", "smooth_abs", "--constraint-transform-epsilon", "0.1", "--current-scale", "100000", "--target-tile-size", "1024", "--source-tile-size", "4320", "--output", str(result_path)], cwd=repo, env=env, check=True)

In [ ]:
import json

result = json.loads(result_path.read_text())
assert result["schema_version"] == 2
assert result["workflow"] == "local_residual_augmented_lagrangian"
assert result["environment"]["jax_backend"] == "gpu"
assert result["cpu"]["execution_platform"] == "cpu"
assert result["gpu"]["execution_platform"] == "gpu"
assert result["solver"]["require_inner_stationarity"]
assert result["solver"]["history_vector_mode"] == "summary"
assert result["solver"]["penalty_update_mode"] == "global"
assert result["method"]["constraint_transform"] == "smooth_abs"
assert result["solver"]["inner_stationarity_relative_tolerance"] == 0.01
assert result["residual_scaling"]["continuation"]["reduction_factor"] == 0.5
assert result["validation_policy"]["target_relative_tolerance"] == 0.10
assert result["scientifically_validated"] == result["scientific_validation"]["passed"]
for backend in ("cpu", "gpu"):
    metrics = result[backend]["final_metrics"]
    assert {"objective", "normalized_normal_field", "coil_constraints"} <= metrics.keys()
    for artifact in result["visualizations"][f"{backend}_final"].values():
        if not isinstance(artifact, str) or not artifact.endswith((".vts", ".vtu")):
            continue
        path = artifact_root / artifact
        assert path.is_file() and path.stat().st_size > 0, path
failed = {name: gate for name, gate in result["acceptance_gates"].items() if not gate["passed"]}
print(json.dumps({"scientifically_validated": result["scientifically_validated"], "all_gates_passed": result["all_gates_passed"], "failed_gates": failed, "diagnostic_checks": result["diagnostic_checks"], "comparison": result["comparison"]}, indent=2))

In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/simsopt-local-residual-al", "zip", artifact_root)
files.download(archive)

## What to send back

Send the downloaded `simsopt-local-residual-al.zip`. A failed scientific gate is still useful data: the archive is downloaded without asserting that every gate passed.